# 🌟 Tutorial 04: MAML - Model-Agnostic Meta-Learning

## El Algoritmo Estrella del Meta-Learning

En este tutorial aprenderás:

- 🎯 Qué es MAML y por qué es revolucionario
- 🔄 Inner loop vs Outer loop optimization
- 📐 Gradientes de segundo orden
- 💻 Implementación completa de MAML

---

## 📖 Parte 1: Teoría

### ¿Qué es MAML?

**MAML** (Finn et al., 2017) es un algoritmo que aprende una **inicialización óptima** de los parámetros del modelo. Esta inicialización está optimizada específicamente para permitir **adaptación rápida** a nuevas tareas con pocos pasos de gradient descent.

### La Gran Idea:

> "No busques parámetros que funcionen bien en promedio, busca parámetros desde los cuales sea fácil aprender."

### Dos Niveles de Optimización:

#### 1️⃣ **Inner Loop** (Task-specific adaptation):
- Para cada tarea $\mathcal{T}_i$, adaptamos los parámetros $\theta$ con K pasos de gradient descent:
$$\theta'_i = \theta - \alpha \nabla_{\theta} \mathcal{L}_{\mathcal{T}_i}^{support}(\theta)$$

#### 2️⃣ **Outer Loop** (Meta-optimization):
- Actualizamos la inicialización $\theta$ para minimizar el loss en query sets después de adaptación:
$$\theta \leftarrow \theta - \beta \nabla_{\theta} \sum_{\mathcal{T}_i} \mathcal{L}_{\mathcal{T}_i}^{query}(\theta'_i)$$

### Visualización:

```
       θ (Meta-parameters)
       |
       |-- Task 1: θ → θ'₁ (inner loop) → evaluate on query
       |-- Task 2: θ → θ'₂ (inner loop) → evaluate on query
       |-- Task 3: θ → θ'₃ (inner loop) → evaluate on query
       |
       Update θ based on all query losses (outer loop)
```

### ¿Por qué funciona?

MAML encuentra un punto en el espacio de parámetros desde el cual:
- Un pequeño paso de gradient descent lleva a buenas soluciones para cualquier tarea
- La "geometría" del loss landscape es favorable para adaptación


---

## 🛠️ Parte 2: Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import sys
sys.path.append('..')

# Importar higher para MAML eficiente
try:
    import higher
    HIGHER_AVAILABLE = True
except ImportError:
    HIGHER_AVAILABLE = False
    print("⚠️  La librería 'higher' no está instalada. Usaremos implementación manual.")

from utils.test_utils import print_success, print_hint, HintSystem, run_test
from utils.data_utils import create_sine_task, set_seed
from utils.visualization import plot_few_shot_results, plot_learning_curves

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Dispositivo: {device}")
print("✅ Setup completo!")

---

## 💻 Parte 3: Modelo para Regresión

In [ ]:
class SineModel(nn.Module):
    """Modelo simple para regresión de funciones seno."""
    
    def __init__(self, hidden_size=40):
        super(SineModel, self).__init__()
        self.fc1 = nn.Linear(1, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

print("✅ Modelo definido!")

---

## 💻 Parte 4: Ejercicio 1 - Inner Loop de MAML

El inner loop adapta el modelo a una tarea específica.

**Tu tarea**: Completa la función del inner loop.

In [ ]:
def inner_loop(model, task, inner_lr=0.01, inner_steps=5):
    """
    Adapta el modelo a una tarea usando K pasos de gradient descent.
    
    Args:
        model: Modelo a adaptar
        task: Diccionario con x_support, y_support
        inner_lr: Learning rate para adaptación
        inner_steps: Número de pasos de gradient descent
    
    Returns:
        adapted_model: Modelo adaptado (copia)
    """
    # TODO: Implementa el inner loop
    # 1. Crea una copia del modelo (usa deepcopy)
    # 2. Crea un optimizador SGD para la copia
    # 3. Entrena la copia por inner_steps pasos en el support set
    # 4. Retorna el modelo adaptado
    
    pass  # TODO: Reemplaza con tu código


# Sistema de pistas
hints_inner = HintSystem([
    "Usa deepcopy(model) para crear una copia independiente del modelo.",
    "El inner loop es entrenamiento estándar: loop de inner_steps con zero_grad, forward, loss, backward, step.",
    "Usa MSELoss para regresión y entrena solo en x_support, y_support.",
    "Código: adapted = deepcopy(model); opt = SGD(adapted.parameters(), lr=inner_lr); for _ in range(inner_steps): [train]; return adapted"
])

In [ ]:
# Para ver pistas
hints_inner.show_hint()

In [ ]:
# ✅ TEST 1: Verificar inner loop

def test_inner_loop():
    model = SineModel()
    task = create_sine_task(k_shot=10, q_query=10)
    
    # Evaluar antes de adaptar
    with torch.no_grad():
        pred_before = model(task['x_query'])
        loss_before = F.mse_loss(pred_before, task['y_query']).item()
    
    # Adaptar
    adapted_model = inner_loop(model, task, inner_lr=0.01, inner_steps=5)
    
    # Evaluar después
    with torch.no_grad():
        pred_after = adapted_model(task['x_query'])
        loss_after = F.mse_loss(pred_after, task['y_query']).item()
    
    # Verificar que el modelo se adaptó
    assert adapted_model is not None, "inner_loop debe retornar un modelo"
    assert loss_after < loss_before, f"Loss debe disminuir después de adaptación (antes: {loss_before:.4f}, después: {loss_after:.4f})"
    
    print_success(f"✅ Inner loop funciona! Loss antes: {loss_before:.4f}, después: {loss_after:.4f}")

run_test(test_inner_loop, "Test de Inner Loop")

---

## 💻 Parte 5: Ejercicio 2 - MAML Completo (Versión Simplificada)

Ahora implementaremos MAML completo. Esta es una versión simplificada que no usa gradientes de segundo orden.

**Tu tarea**: Completa el meta-training loop.

In [ ]:
def train_maml_simple(model, n_iterations=1000, meta_batch_size=4, 
                      inner_lr=0.01, outer_lr=0.001, inner_steps=5):
    """
    Entrena modelo con MAML (versión simplificada, sin gradientes de 2do orden).
    
    Args:
        model: Modelo a meta-entrenar
        n_iterations: Número de meta-iteraciones
        meta_batch_size: Número de tareas por iteración
        inner_lr: LR para inner loop
        outer_lr: LR para outer loop
        inner_steps: Pasos de adaptación por tarea
    
    Returns:
        meta_losses: Lista de losses por iteración
    """
    meta_optimizer = optim.Adam(model.parameters(), lr=outer_lr)
    criterion = nn.MSELoss()
    
    meta_losses = []
    
    print(f"🚀 Meta-entrenando por {n_iterations} iteraciones...\n")
    
    for iteration in range(n_iterations):
        meta_loss = 0.0
        
        # TODO: Implementa el meta-training loop
        # Para cada tarea en el meta-batch:
        #   1. Sample una tarea
        #   2. Adapta el modelo (inner loop)
        #   3. Evalúa el modelo adaptado en el query set
        #   4. Acumula el loss
        # 
        # Después de todas las tareas:
        #   5. Calcula el promedio del meta-loss
        #   6. Actualiza el modelo original (outer loop)
        
        pass  # TODO: Reemplaza con tu código
        
        if (iteration + 1) % 100 == 0:
            avg_loss = np.mean(meta_losses[-100:])
            print(f"Iteración {iteration+1}/{n_iterations} - Meta-Loss: {avg_loss:.4f}")
    
    return meta_losses


# Sistema de pistas
hints_maml = HintSystem([
    "El loop externo itera sobre iteraciones, el interno sobre meta_batch_size tareas.",
    "Para cada tarea: task = create_sine_task(); adapted = inner_loop(model, task); query_loss = criterion(adapted(x_query), y_query).",
    "Acumula meta_loss y al final: meta_loss /= meta_batch_size; meta_optimizer.zero_grad(); meta_loss.backward(); meta_optimizer.step().",
    "Importante: Necesitas mantener el grafo computacional entre inner y outer loop para que backward funcione."
])

In [ ]:
# Para ver pistas
hints_maml.show_hint()

---

## 📊 Parte 6: MAML con la Librería 'higher' (Implementación Correcta)

La librería `higher` nos permite implementar MAML correctamente con gradientes de segundo orden.

In [ ]:
if HIGHER_AVAILABLE:
    def train_maml_higher(model, n_iterations=1000, meta_batch_size=4,
                          inner_lr=0.01, outer_lr=0.001, inner_steps=5):
        """
        MAML con higher (gradientes de segundo orden correctos).
        """
        meta_optimizer = optim.Adam(model.parameters(), lr=outer_lr)
        criterion = nn.MSELoss()
        
        meta_losses = []
        
        print(f"🚀 Meta-entrenando con higher por {n_iterations} iteraciones...\n")
        
        for iteration in range(n_iterations):
            meta_optimizer.zero_grad()
            meta_loss = 0.0
            
            for _ in range(meta_batch_size):
                # Sample task
                task = create_sine_task(k_shot=10, q_query=10)
                x_support, y_support = task['x_support'], task['y_support']
                x_query, y_query = task['x_query'], task['y_query']
                
                # Inner loop con higher
                with higher.innerloop_ctx(model, meta_optimizer, 
                                         copy_initial_weights=False) as (fmodel, diffopt):
                    # Adaptación (inner loop)
                    for _ in range(inner_steps):
                        support_pred = fmodel(x_support)
                        support_loss = criterion(support_pred, y_support)
                        diffopt.step(support_loss)
                    
                    # Evaluación en query (outer loop)
                    query_pred = fmodel(x_query)
                    query_loss = criterion(query_pred, y_query)
                    meta_loss += query_loss
            
            # Meta-update
            meta_loss = meta_loss / meta_batch_size
            meta_loss.backward()
            meta_optimizer.step()
            
            meta_losses.append(meta_loss.item())
            
            if (iteration + 1) % 100 == 0:
                avg_loss = np.mean(meta_losses[-100:])
                print(f"Iteración {iteration+1}/{n_iterations} - Meta-Loss: {avg_loss:.4f}")
        
        return meta_losses
    
    # Entrenar con MAML
    maml_model = SineModel().to(device)
    maml_losses = train_maml_higher(maml_model, n_iterations=500, meta_batch_size=4)
    
    print("\n✅ Meta-entrenamiento completado!")
else:
    print("⚠️  Instala 'higher' para entrenar con la implementación correcta de MAML.")
    print("   Ejecuta: pip install higher")

---

## 📊 Parte 7: Evaluación y Comparación

Comparemos MAML con entrenamiento tradicional.

In [ ]:
if HIGHER_AVAILABLE:
    # Crear nueva tarea de test
    test_task = create_sine_task(k_shot=10, q_query=50)
    
    # 1. MAML: Adaptación rápida
    maml_adapted = deepcopy(maml_model)
    optimizer = optim.SGD(maml_adapted.parameters(), lr=0.01)
    maml_curve = []
    
    # Evaluación inicial
    with torch.no_grad():
        pred = maml_adapted(test_task['x_query'])
        loss = F.mse_loss(pred, test_task['y_query']).item()
        maml_curve.append(loss)
    
    # Adaptar por 50 pasos
    for step in range(50):
        optimizer.zero_grad()
        pred = maml_adapted(test_task['x_support'])
        loss = F.mse_loss(pred, test_task['y_support'])
        loss.backward()
        optimizer.step()
        
        # Evaluar en query
        with torch.no_grad():
            pred = maml_adapted(test_task['x_query'])
            loss = F.mse_loss(pred, test_task['y_query']).item()
            maml_curve.append(loss)
    
    # 2. ML Tradicional: Desde cero
    scratch_model = SineModel().to(device)
    optimizer = optim.SGD(scratch_model.parameters(), lr=0.01)
    scratch_curve = []
    
    with torch.no_grad():
        pred = scratch_model(test_task['x_query'])
        loss = F.mse_loss(pred, test_task['y_query']).item()
        scratch_curve.append(loss)
    
    for step in range(50):
        optimizer.zero_grad()
        pred = scratch_model(test_task['x_support'])
        loss = F.mse_loss(pred, test_task['y_support'])
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            pred = scratch_model(test_task['x_query'])
            loss = F.mse_loss(pred, test_task['y_query']).item()
            scratch_curve.append(loss)
    
    # Visualizar
    plt.figure(figsize=(12, 5))
    plt.plot(scratch_curve, label='ML Tradicional (Random Init)', color='red', linewidth=2)
    plt.plot(maml_curve, label='MAML (Meta-learned Init)', color='green', linewidth=2)
    plt.xlabel('Pasos de Adaptación', fontsize=12)
    plt.ylabel('Loss en Query Set', fontsize=12)
    plt.title('MAML vs ML Tradicional: Velocidad de Adaptación', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print(f"\n📊 Comparación:")
    print(f"  ML Tradicional - Loss inicial: {scratch_curve[0]:.4f}, Final: {scratch_curve[-1]:.4f}")
    print(f"  MAML - Loss inicial: {maml_curve[0]:.4f}, Final: {maml_curve[-1]:.4f}")
    print(f"\n🎯 MAML es {scratch_curve[10] / maml_curve[10]:.1f}x mejor después de 10 pasos!")

---

## 🎓 Resumen y Conclusiones

### ✅ Lo que aprendiste:

1. **MAML** optimiza la inicialización para adaptación rápida
2. Usa **dos niveles de optimización**: inner loop (adaptación) y outer loop (meta-learning)
3. Requiere **gradientes de segundo orden** para funcionar correctamente
4. La librería **higher** facilita la implementación correcta

### 🔍 Ventajas de MAML:

- ✅ Model-agnostic (funciona con cualquier arquitectura)
- ✅ Adaptación extremadamente rápida (1-5 pasos)
- ✅ No requiere estructuras especiales
- ✅ State-of-the-art en many Few-Shot tasks

### ⚖️ Limitaciones:

- ⚠️ Computacionalmente costoso (gradientes de 2do orden)
- ⚠️ Puede ser inestable sin careful tuning
- ⚠️ Requiere suficiente diversidad en las tareas de entrenamiento

### 🚀 Próximo Tutorial:

En el **Tutorial 05** veremos **Meta-Learning con Memoria**, donde usamos RNNs para que el modelo "recuerde" cómo adaptar sus parámetros.

---

## 🎉 ¡Felicidades!

Has implementado MAML, uno de los algoritmos más importantes e influyentes en Meta-Learning!
